# Context-Aware Misinformation Detection — DistilBERT Classifier

**Label convention:** `0 = fake`, `1 = true` (matches your CSV).

**Note on title + content:** earlier versions of this notebook combined `title` and `content` at train time, since headlines carry a strong, cheap signal that pure body text dilutes. Your current `bd_combined_news.csv` already has that merge done upstream — a single `content` column holding title + body — so this version reads `content` directly and skips the in-notebook combine step. If you ever regenerate the CSV with `title` and `content` split apart again, reintroduce a concatenation step (`title + ". " + content`) before Section 3.

## 1. Install & import

In [ ]:
# Colab periodically ships torch/torchvision/torchaudio built against
# different CUDA versions or mismatched builds, which throws CUDA-version
# warnings and can even break unrelated imports (e.g. the `datasets` library
# probing torchvision.io.VideoReader for video-tensor support). This
# notebook is text-only (DistilBERT) and never touches images or audio, so
# the simplest, most durable fix is to remove both rather than chase a
# matching build every time Colab's base image shifts.
!pip uninstall -y -q torchaudio torchvision
!pip install -q transformers datasets accelerate evaluate scikit-learn seaborn matplotlib


In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, classification_report,
    roc_curve, auc, precision_recall_curve
)

from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from datasets import Dataset

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)


## 2. Mount Drive & load data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

INPUT_CSV = "/content/drive/MyDrive/Colab Notebooks/Context-Aware Misinformation Detection/BigDataSet/clean/bd_combined_news.csv"
OUTPUT_DIR = "/content/drive/MyDrive/Colab Notebooks/Context-Aware Misinformation Detection/BigDataSet/DistilBERT/pickle"
os.makedirs(OUTPUT_DIR, exist_ok=True)

df = pd.read_csv(INPUT_CSV)
print(df.shape)
df.head()


In [ ]:
# Basic cleanup: drop rows with missing essential fields, ensure label is int
df = df.dropna(subset=["content", "label"]).copy()
df["label"] = df["label"].astype(int)
df["content"] = df["content"].astype(str).str.strip()

print("Rows after cleanup:", len(df))
print(df["label"].value_counts(normalize=True))

plt.figure(figsize=(4, 4))
sns.countplot(x="label", data=df)
plt.title("Label distribution (0=fake, 1=true)")
plt.xlabel("label")
plt.ylabel("count")
plt.show()


## 2.1 Write a reusable cleaning module (so it can live inside the pickle)

Pickle only stores a **reference** to a class or function's module + name, not its actual code — so for the pickle to reload correctly later (fresh Colab session, or someone else's machine), the cleaning function and the predictor class need to live in a real, importable `.py` file rather than be defined directly in a notebook cell. This writes that module to disk and imports it; the next cell copies it into `OUTPUT_DIR` so the `.py` and the `.pkl` that depends on it always travel together.

In [ ]:
%%writefile inference_module.py
import re
import html
import torch


def clean_text(text):
    """Raw-text cleanup: unescape HTML entities, strip tags/URLs, collapse whitespace."""
    if text is None:
        return ""
    text = str(text)
    text = html.unescape(text)
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


class FakeNewsPredictor:
    """Bundles cleaning + tokenizer + model so a single pickle goes
    straight from raw content text to a prediction."""

    def __init__(self, model, tokenizer, max_length, label_map=None):
        self.model = model
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.label_map = label_map or {0: "fake", 1: "true"}

    def predict(self, text, device=None):
        text = clean_text(text)
        dev = device or next(self.model.parameters()).device
        inputs = self.tokenizer(
            text, truncation=True, padding="max_length",
            max_length=self.max_length, return_tensors="pt"
        ).to(dev)
        self.model.eval()
        with torch.no_grad():
            logits = self.model(**inputs).logits
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
        label = int(probs.argmax())
        return {
            "label": label,
            "meaning": self.label_map[label],
            "confidence": float(probs[label]),
        }

    def __call__(self, text):
        return self.predict(text)


In [ ]:
import shutil
import sys
import importlib

module_dest = os.path.join(OUTPUT_DIR, "inference_module.py")
shutil.copy("inference_module.py", module_dest)

if OUTPUT_DIR not in sys.path:
    sys.path.insert(0, OUTPUT_DIR)

import inference_module
importlib.reload(inference_module)
from inference_module import clean_text, FakeNewsPredictor

print("Cleaning module ready; copied to:", module_dest)


## 3. Clean the content field

`content` already has title + body merged upstream, but it can still carry HTML remnants, stray URLs, or irregular whitespace from scraping. Route it through the same `clean_text()` that's bundled into the pickle, so training sees exactly what inference will see later.

In [ ]:
df["content"] = df["content"].apply(clean_text)
df = df[["content", "label"]]
df.head()


## 4. Stratified train / val / test split

Per your spec: `test_size=0.3`, `stratify=True` for the held-out test set. I additionally carve a small stratified validation slice out of the remaining 70% so the Trainer can do early stopping / pick the best checkpoint without ever touching the test set.

In [ ]:
train_val_df, test_df = train_test_split(
    df, test_size=0.3, stratify=df["label"], random_state=SEED
)

train_df, val_df = train_test_split(
    train_val_df, test_size=0.1, stratify=train_val_df["label"], random_state=SEED
)

print("train:", train_df.shape, "val:", val_df.shape, "test:", test_df.shape)
for name, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(name, d["label"].value_counts(normalize=True).to_dict())


## 5. Tokenize

In [ ]:
MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 256  # bump to 512 if you want the full article body; slower + more memory

tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)

def to_hf_dataset(d):
    ds = Dataset.from_pandas(d[["content", "label"]].reset_index(drop=True))
    return ds.map(
        lambda batch: tokenizer(
            batch["content"], truncation=True, padding="max_length", max_length=MAX_LENGTH
        ),
        batched=True,
    )

train_ds = to_hf_dataset(train_df)
val_ds = to_hf_dataset(val_df)
test_ds = to_hf_dataset(test_df)

train_ds = train_ds.rename_column("label", "labels")
val_ds = val_ds.rename_column("label", "labels")
test_ds = test_ds.rename_column("label", "labels")

cols = ["input_ids", "attention_mask", "labels"]
train_ds.set_format(type="torch", columns=cols)
val_ds.set_format(type="torch", columns=cols)
test_ds.set_format(type="torch", columns=cols)


## 6. Model + Trainer

Parameters below are the widely-used, well-behaved defaults for fine-tuning DistilBERT on binary text classification (2–4 epochs, lr 2e-5, linear warmup, weight decay 0.01). They're a strong starting point for a few hundred thousand rows; if you have GPU budget to spare, a quick Optuna/Ray Tune sweep over `learning_rate` and `num_train_epochs` is the natural next step (see the "extend" notes at the end).

In [ ]:
model = DistilBertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.to(device)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary", zero_division=0
    )
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}

training_args = TrainingArguments(
    output_dir="/content/dbert_ckpts",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_ratio=0.1,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)


In [ ]:
train_result = trainer.train()


## 7. Training curves

In [ ]:
log_history = trainer.state.log_history
train_loss = [(e["step"], e["loss"]) for e in log_history if "loss" in e]
eval_loss = [(e["epoch"], e["eval_loss"]) for e in log_history if "eval_loss" in e]
eval_f1 = [(e["epoch"], e["eval_f1"]) for e in log_history if "eval_f1" in e]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

if train_loss:
    steps, losses = zip(*train_loss)
    axes[0].plot(steps, losses, label="train loss")
if eval_loss:
    epochs, losses = zip(*eval_loss)
    axes[0].plot([e for e in epochs], losses, marker="o", label="val loss")
axes[0].set_xlabel("step / epoch")
axes[0].set_ylabel("loss")
axes[0].set_title("Loss")
axes[0].legend()

if eval_f1:
    epochs, f1s = zip(*eval_f1)
    axes[1].plot(epochs, f1s, marker="o", color="green")
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("F1")
axes[1].set_title("Validation F1")

plt.tight_layout()
plt.show()


## 8. Final evaluation on the held-out test set

In [ ]:
pred_output = trainer.predict(test_ds)
logits = pred_output.predictions
probs = torch.softmax(torch.tensor(logits), dim=1).numpy()[:, 1]  # P(label=1 / true)
y_pred = np.argmax(logits, axis=-1)
y_true = pred_output.label_ids

print(classification_report(y_true, y_pred, target_names=["fake (0)", "true (1)"]))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["fake", "true"], yticklabels=["fake", "true"], ax=axes[0])
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Actual")
axes[0].set_title("Confusion Matrix")

# ROC curve
fpr, tpr, _ = roc_curve(y_true, probs)
roc_auc = auc(fpr, tpr)
axes[1].plot(fpr, tpr, label=f"AUC = {roc_auc:.3f}")
axes[1].plot([0, 1], [0, 1], linestyle="--", color="gray")
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("ROC Curve")
axes[1].legend()

# Precision-Recall curve
prec, rec, _ = precision_recall_curve(y_true, probs)
axes[2].plot(rec, prec)
axes[2].set_xlabel("Recall")
axes[2].set_ylabel("Precision")
axes[2].set_title("Precision-Recall Curve")

plt.tight_layout()
plt.show()


## 9. Save the model — bundled with cleaning, ready for raw-text inference

The pickle holds a single `FakeNewsPredictor` object: `clean_text()` + tokenizer + trained model, all together. Whoever loads it just calls `.predict(text)` on **raw, uncleaned** content — they never touch tokenization or preprocessing themselves.

Two things still worth keeping in mind:
1. `inference_module.py` (saved right next to the pickle) must be importable wherever you unpickle this — pickle stores a reference to the class, not its code, so keep the two files together and add their folder to `sys.path` before loading.
2. `save_pretrained` below is still saved as a fallback — the safest way to recover just the raw model weights if a library version bump ever breaks the pickle.

In [ ]:
# Fallback: robust HF format (recommended if the pickle ever breaks across versions)
hf_dir = os.path.join(OUTPUT_DIR, "hf_model")
os.makedirs(hf_dir, exist_ok=True)
trainer.save_model(hf_dir)
tokenizer.save_pretrained(hf_dir)

# Bundle cleaning + tokenizer + model into one object and pickle it
predictor = FakeNewsPredictor(
    model=model,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH,
    label_map={0: "fake", 1: "true"},
)

pickle_path = os.path.join(OUTPUT_DIR, "distilbert_fake_news.pkl")
with open(pickle_path, "wb") as f:
    pickle.dump(predictor, f)

print("Saved HF fallback model to:", hf_dir)
print("Saved bundled predictor pickle to:", pickle_path)
print("Cleaning module lives at:", os.path.join(OUTPUT_DIR, "inference_module.py"))


## 10. Inference helper (reload the bundled pickle, predict on raw text)

This is what a fresh session — yours next week, or a teammate's — actually needs to run:

In [ ]:
import sys
import pickle

OUTPUT_DIR = "/content/drive/MyDrive/Colab Notebooks/Context-Aware Misinformation Detection/BigDataSet/DistilBERT/pickle"
pickle_path = os.path.join(OUTPUT_DIR, "distilbert_fake_news.pkl")

# inference_module.py must be importable before unpickling, since the pickle
# only stores a reference to FakeNewsPredictor, not its code.
if OUTPUT_DIR not in sys.path:
    sys.path.insert(0, OUTPUT_DIR)
from inference_module import FakeNewsPredictor  # noqa: F401 (import needed for unpickling)

with open(pickle_path, "rb") as f:
    loaded_predictor = pickle.load(f)

# Raw, uncleaned content goes in directly — cleaning happens inside .predict()
example = loaded_predictor.predict(
    "<p>BREAKING!!! You WON'T believe this... http://example.com</p>"
)
print(example)
